# GSR/EDA preprocessing and feature extraction pipeline

This notebook provides a cleaned and translated version of the GSR/EDA processing workflow. It preserves the original logic of subject-level and stimulus-level processing while removing duplicated cells, syntax errors, debugging outputs, and Spanish comments.


## 1. Imports, configuration, and expected folder structure

Expected raw data structure:

```text
RAW_DATA_DIR/
    1/Session01/TestS01R000/gsr_signal.csv
    1/Session01/TestS01R000/stimulus.csv
    2/Session01/TestS01R000/gsr_signal.csv
    ...
```

Update the paths in `ProcessingConfig` before running the notebook.


In [ ]:
from __future__ import annotations

import logging
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

@dataclass
class ProcessingConfig:
    """Configuration parameters for the GSR/EDA processing pipeline."""

    # Generic, portable paths
    raw_data_dir: Path = Path("data/raw")
    output_dir: Path = Path("data/processed")

    sampling_rate: int = 32
    subject_ids: Iterable[int] = range(1, 52)
    stimulus_code_for_images: int = 2
    stimulus_duration_sec: float = 6.0
    timestamp_scale: float = 1e6  # timestamps are assumed to be in microseconds
    save_plots: bool = True

    def __post_init__(self):
        # Ensure output directory exists
        self.output_dir.mkdir(parents=True, exist_ok=True)


GSR_COLUMNS = [
    "GSR",
    "BVP",
    "ACC_x",
    "ACC_y",
    "ACC_z",
    "sequence_number",
    "timestamp_reception",
    "timestamp_corrected",
]

STIMULUS_COLUMNS = ["timestamp", "stimulus_code", "value"]


## 2. Helper functions


In [ ]:
# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------

def configure_logging() -> None:
    """Configure informative logging messages."""
    logging.basicConfig(
        level=logging.INFO,
        format="%(levelname)s | %(message)s",
    )


def find_test_folder(session_path: Path) -> Optional[Path]:
    """Return the first folder whose name starts with 'TestS01R'."""
    if not session_path.exists():
        return None

    test_folders = sorted(
        folder for folder in session_path.iterdir()
        if folder.is_dir() and folder.name.startswith("TestS01R")
    )
    return test_folders[0] if test_folders else None


def read_gsr_file(gsr_path: Path) -> pd.DataFrame:
    """Read a GSR CSV file and standardise column names."""
    gsr_df = pd.read_csv(gsr_path, header=None, names=GSR_COLUMNS)

    gsr_df["GSR"] = pd.to_numeric(gsr_df["GSR"], errors="coerce")
    gsr_df["timestamp_corrected"] = pd.to_numeric(
        gsr_df["timestamp_corrected"], errors="coerce"
    )

    gsr_df = gsr_df.replace([np.inf, -np.inf], np.nan)
    gsr_df = gsr_df.dropna(subset=["GSR", "timestamp_corrected"]).reset_index(drop=True)

    return gsr_df


def read_stimulus_file(stimulus_path: Path) -> pd.DataFrame:
    """Read the stimulus CSV file and standardise column names."""
    stimulus_df = pd.read_csv(stimulus_path, header=None, names=STIMULUS_COLUMNS)
    stimulus_df["timestamp"] = pd.to_numeric(stimulus_df["timestamp"], errors="coerce")
    stimulus_df["stimulus_code"] = pd.to_numeric(
        stimulus_df["stimulus_code"], errors="coerce"
    )
    stimulus_df = stimulus_df.dropna(subset=["timestamp", "stimulus_code"])
    return stimulus_df.reset_index(drop=True)


def is_valid_signal(signal: pd.Series) -> bool:
    """Check whether the EDA signal has enough valid variability to process."""
    return not signal.empty and signal.nunique(dropna=True) > 1


def process_eda_signal(gsr_signal: pd.Series, sampling_rate: int) -> pd.DataFrame:
    """Clean and decompose the GSR/EDA signal using NeuroKit2."""
    try:
        eda_signals, _ = nk.eda_process(gsr_signal, sampling_rate=sampling_rate)
        return eda_signals
    except Exception as exc:
        logging.warning("EDA processing failed: %s", exc)
        return pd.DataFrame()
    return eda_signals


def safe_find_scr_peaks(
    phasic_signal: pd.Series,
    sampling_rate: int,
) -> dict[str, np.ndarray]:
    """Detect SCR onsets, peaks, and heights with safeguards for invalid segments."""
    phasic_signal = pd.Series(phasic_signal).replace([np.inf, -np.inf], np.nan).dropna()

    empty_result = {
        "SCR_Onsets": np.array([], dtype=int),
        "SCR_Peaks": np.array([], dtype=int),
        "SCR_Height": np.array([], dtype=float),
    }

    if phasic_signal.empty or phasic_signal.nunique(dropna=True) <= 1:
        return empty_result

    try:
        peaks = nk.eda_findpeaks(phasic_signal, sampling_rate=sampling_rate)
    except (ValueError, IndexError, TypeError):
        return empty_result

    onsets = np.asarray(peaks.get("SCR_Onsets", []), dtype=float)
    scr_peaks = np.asarray(peaks.get("SCR_Peaks", []), dtype=float)
    heights = np.asarray(peaks.get("SCR_Height", []), dtype=float)

    min_len = min(len(onsets), len(scr_peaks), len(heights))
    if min_len == 0:
        return empty_result

    onsets = onsets[:min_len]
    scr_peaks = scr_peaks[:min_len]
    heights = heights[:min_len]

    valid = (
        ~np.isnan(onsets)
        & ~np.isnan(scr_peaks)
        & ~np.isnan(heights)
        & (onsets >= 0)
        & (scr_peaks >= 0)
    )

    return {
        "SCR_Onsets": onsets[valid].astype(int),
        "SCR_Peaks": scr_peaks[valid].astype(int),
        "SCR_Height": heights[valid].astype(float),
    }


def extract_subject_level_scr_features(
    subject_id: str,
    eda_signals: pd.DataFrame,
    sampling_rate: int,
) -> pd.DataFrame:
    """Extract SCR features from the full phasic EDA signal for one subject."""
    scr = safe_find_scr_peaks(eda_signals["EDA_Phasic"], sampling_rate=sampling_rate)

    rows = []
    for onset_idx, peak_idx, amplitude in zip(
        scr["SCR_Onsets"], scr["SCR_Peaks"], scr["SCR_Height"]
    ):
        rows.append(
            {
                "Subject": subject_id,
                "Onset_index": onset_idx,
                "Peak_index": peak_idx,
                "Peak_time_sec": peak_idx / sampling_rate,
                "Latency_sec": (peak_idx - onset_idx) / sampling_rate,
                "Amplitude": amplitude,
            }
        )

    return pd.DataFrame(rows)


def extract_stimulus_level_scr_features(
    subject_id: str,
    gsr_df: pd.DataFrame,
    stimulus_df: pd.DataFrame,
    eda_signals: pd.DataFrame,
    config: ProcessingConfig,
) -> pd.DataFrame:
    """Extract SCR features from image-locked EDA segments."""
    image_events = stimulus_df[
        stimulus_df["stimulus_code"] == config.stimulus_code_for_images
    ].reset_index(drop=True)

    if image_events.empty:
        return pd.DataFrame()

    first_gsr_timestamp = gsr_df["timestamp_corrected"].iloc[0]
    rows = []

    for event_index, event in image_events.iterrows():
        start_sec = (event["timestamp"] - first_gsr_timestamp) / config.timestamp_scale
        end_sec = start_sec + config.stimulus_duration_sec

        time_vector = eda_signals.index / config.sampling_rate
        segment = eda_signals[
            (time_vector >= start_sec) & (time_vector < end_sec)
        ].copy()
        segment.reset_index(drop=True, inplace=True)

        if segment.empty or "EDA_Phasic" not in segment.columns:
            continue

        scr = safe_find_scr_peaks(segment["EDA_Phasic"], config.sampling_rate)

        for onset_idx, peak_idx, amplitude in zip(
            scr["SCR_Onsets"], scr["SCR_Peaks"], scr["SCR_Height"]
        ):
            rows.append(
                {
                    "Subject": subject_id,
                    "Stimulus_index": event_index + 1,
                    "Stimulus_value": event.get("value", np.nan),
                    "Stimulus_start_sec": start_sec,
                    "Stimulus_end_sec": end_sec,
                    "Onset_index": onset_idx,
                    "Peak_index": peak_idx,
                    "Peak_time_sec": peak_idx / config.sampling_rate,
                    "Latency_sec": (peak_idx - onset_idx) / config.sampling_rate,
                    "Amplitude": amplitude,
                }
            )

    return pd.DataFrame(rows)


def plot_subject_phasic_eda(
    subject_id: str,
    eda_signals: pd.DataFrame,
    features_df: pd.DataFrame,
    sampling_rate: int,
    output_path: Path,
) -> None:
    """Save a plot of the full phasic EDA signal with SCR onsets and peaks."""
    plt.figure(figsize=(12, 4))
    time_sec = eda_signals.index / sampling_rate
    plt.plot(time_sec, eda_signals["EDA_Phasic"], label="Phasic EDA")

    if not features_df.empty:
        onset_idx = features_df["Onset_index"].astype(int).to_numpy()
        peak_idx = features_df["Peak_index"].astype(int).to_numpy()

        onset_idx = onset_idx[onset_idx < len(eda_signals)]
        peak_idx = peak_idx[peak_idx < len(eda_signals)]

        if len(onset_idx) > 0:
            plt.scatter(
                onset_idx / sampling_rate,
                eda_signals["EDA_Phasic"].iloc[onset_idx],
                marker="o",
                label="SCR onset",
            )
        if len(peak_idx) > 0:
            plt.scatter(
                peak_idx / sampling_rate,
                eda_signals["EDA_Phasic"].iloc[peak_idx],
                marker="x",
                label="SCR peak",
            )

    plt.xlabel("Time (s)")
    plt.ylabel("EDA / GSR (µS)")
    plt.title(f"Phasic EDA with SCR onsets and peaks - Subject {subject_id}")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_stimulus_segments(
    subject_id: str,
    gsr_df: pd.DataFrame,
    stimulus_df: pd.DataFrame,
    eda_signals: pd.DataFrame,
    config: ProcessingConfig,
    output_path: Path,
) -> None:
    """Save one plot per image stimulus, showing tonic and phasic EDA."""
    image_events = stimulus_df[
        stimulus_df["stimulus_code"] == config.stimulus_code_for_images
    ].reset_index(drop=True)

    if image_events.empty:
        return

    first_gsr_timestamp = gsr_df["timestamp_corrected"].iloc[0]
    n_events = len(image_events)
    fig, axes = plt.subplots(
        n_events,
        1,
        figsize=(10, max(3, 3 * n_events)),
        sharex=False,
    )

    if n_events == 1:
        axes = [axes]

    for ax, (_, event) in zip(axes, image_events.iterrows()):
        start_sec = (event["timestamp"] - first_gsr_timestamp) / config.timestamp_scale
        end_sec = start_sec + config.stimulus_duration_sec

        time_vector = eda_signals.index / config.sampling_rate
        segment = eda_signals[
            (time_vector >= start_sec) & (time_vector < end_sec)
        ].copy()
        segment.reset_index(drop=True, inplace=True)

        if segment.empty:
            ax.set_title(f"Subject {subject_id} - Empty segment")
            ax.set_ylabel("EDA / GSR (µS)")
            continue

        segment_time = segment.index / config.sampling_rate

        ax.plot(segment_time, segment["EDA_Tonic"], label="Tonic EDA")
        ax.plot(segment_time, segment["EDA_Phasic"], label="Phasic EDA")

        scr = safe_find_scr_peaks(segment["EDA_Phasic"], config.sampling_rate)
        peak_idx = scr["SCR_Peaks"]

        if len(peak_idx) > 0:
            valid_peaks = peak_idx[peak_idx < len(segment)]

            if len(valid_peaks) > 0:
                ax.scatter(
                    valid_peaks / config.sampling_rate,
                    segment["EDA_Phasic"].iloc[valid_peaks],
                    marker="x",
                    label="SCR peak",
                )

        ax.set_title(f"Subject {subject_id} - Stimulus {event.get('value', '')}")
        ax.set_ylabel("EDA / GSR (µS)")
        ax.legend(loc="best")

    axes[-1].set_xlabel("Time within segment (s)")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


## 3. Main processing workflow


In [ ]:
# -----------------------------------------------------------------------------
# Main processing workflow
# -----------------------------------------------------------------------------

def process_all_subjects(config: ProcessingConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run the full GSR/EDA processing pipeline for all configured subjects."""
    csv_dir = config.output_dir / "CSVs"
    pdf_dir = config.output_dir / "PDFs"
    csv_dir.mkdir(parents=True, exist_ok=True)
    pdf_dir.mkdir(parents=True, exist_ok=True)

    all_subject_features = []
    all_stimulus_features = []

    for subject_number in config.subject_ids:
        subject_id = str(subject_number)
        session_path = config.raw_data_dir / subject_id / "Session01"
        test_folder = find_test_folder(session_path)

        if test_folder is None:
            logging.info("Subject %s: Test folder not found, skipping.", subject_id)
            continue

        gsr_path = test_folder / "gsr_signal.csv"
        stimulus_path = test_folder / "stimulus.csv"

        if not gsr_path.exists():
            logging.info("Subject %s: GSR file not found, skipping.", subject_id)
            continue

        if not stimulus_path.exists():
            logging.info("Subject %s: stimulus file not found, skipping.", subject_id)
            continue

        try:
            gsr_df = read_gsr_file(gsr_path)
            stimulus_df = read_stimulus_file(stimulus_path)
        except Exception as exc:  # noqa: BLE001
            logging.warning("Subject %s: failed to read files (%s).", subject_id, exc)
            continue

        if not is_valid_signal(gsr_df["GSR"]):
            logging.info("Subject %s: invalid or constant GSR signal, skipping.", subject_id)
            continue

        try:
            eda_signals = process_eda_signal(gsr_df["GSR"], config.sampling_rate)
        except Exception as exc:  # noqa: BLE001
            logging.warning("Subject %s: EDA processing failed (%s).", subject_id, exc)
            continue

        subject_features = extract_subject_level_scr_features(
            subject_id, eda_signals, config.sampling_rate
        )
        stimulus_features = extract_stimulus_level_scr_features(
            subject_id, gsr_df, stimulus_df, eda_signals, config
        )

        if not subject_features.empty:
            subject_features.to_csv(csv_dir / f"Subject_{subject_id}_EDA_peaks.csv", index=False)
            all_subject_features.append(subject_features)

        if not stimulus_features.empty:
            stimulus_features.to_csv(
                csv_dir / f"Subject_{subject_id}_EDA_peaks_by_stimulus.csv",
                index=False,
            )
            all_stimulus_features.append(stimulus_features)

        if config.save_plots:
            plot_subject_phasic_eda(
                subject_id,
                eda_signals,
                subject_features,
                config.sampling_rate,
                pdf_dir / f"Subject_{subject_id}_EDA_phasic.pdf",
            )
            plot_stimulus_segments(
                subject_id,
                gsr_df,
                stimulus_df,
                eda_signals,
                config,
                pdf_dir / f"Subject_{subject_id}_EDA_by_stimulus.pdf",
            )

        logging.info("Subject %s: processing completed.", subject_id)

    subject_level_df = (
        pd.concat(all_subject_features, ignore_index=True)
        if all_subject_features else pd.DataFrame()
    )
    stimulus_level_df = (
        pd.concat(all_stimulus_features, ignore_index=True)
        if all_stimulus_features else pd.DataFrame()
    )

    subject_level_df.to_csv(csv_dir / "EDA_subject_level_features_all.csv", index=False)
    stimulus_level_df.to_csv(csv_dir / "EDA_stimulus_level_features_all.csv", index=False)

    return subject_level_df, stimulus_level_df


## 4. Run the pipeline

This cell processes all configured subjects, saves individual and aggregated CSV files, and optionally exports PDF plots for visual inspection.


In [ ]:
configure_logging()

config = ProcessingConfig(
    raw_data_dir=Path("data/raw"),
    output_dir=Path("data/processed"),
    sampling_rate=32,
    subject_ids=range(1, 52),
    stimulus_code_for_images=2,
    stimulus_duration_sec=6.0,
    save_plots=True,
)

subject_features, stimulus_features = process_all_subjects(config)

print(f"Subject-level feature rows: {len(subject_features)}")
print(f"Stimulus-level feature rows: {len(stimulus_features)}")


## 5. Reproducibility notes

- Check that the sampling rate matches the acquisition device configuration.
- The code assumes timestamps are expressed in microseconds; change `timestamp_scale` if a different unit is used.
- `stimulus_code_for_images = 2` follows the coding used in the original experimental files.
- The pipeline exports both full-recording SCR features and stimulus-locked SCR features.
